# Hong Kong AQI Short-Term Prediction
## COMP5152 Group Project

## 1. Introduction

This notebook presents an analysis of Hong Kong Air Quality Index (AQI) prediction using machine learning and time series methods.

**Objectives:**
- Predict AQI for the next 1-3 days
- Provide health recommendations for residents
- Compare multiple modeling approaches

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Load data
DATA_DIR = 'data'
processed_df = pd.read_csv(f'{DATA_DIR}/processed_aqi_data.csv', index_col=0, parse_dates=True)
print(f"Data shape: {processed_df.shape}")
print(f"Date range: {processed_df.index.min()} to {processed_df.index.max()}")

## 2. Exploratory Data Analysis

In [ ]:
# Data overview
processed_df.head()

In [ ]:
# Data statistics
processed_df.describe()

In [ ]:
# Missing values
missing = processed_df.isnull().sum()
print("Missing values:")
print(missing[missing > 0])

In [ ]:
# Plot PM2.5 time series
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(processed_df.index, processed_df['pm2_5'], alpha=0.7, linewidth=0.5)
ax.set_title('PM2.5 Concentration Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('PM2.5 (μg/m³)')
plt.tight_layout()
plt.show()

In [ ]:
# Daily patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By hour
hourly = processed_df.groupby(processed_df.index.hour)['pm2_5'].mean()
axes[0].bar(hourly.index, hourly.values, color='steelblue')
axes[0].set_title('Average PM2.5 by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('PM2.5 (μg/m³)')

# By month
monthly = processed_df.groupby(processed_df.index.month)['pm2_5'].mean()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].bar(range(1, 13), monthly.values, color='coral')
axes[1].set_title('Average PM2.5 by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('PM2.5 (μg/m³)')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(months, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = ['pm2_5', 'pm10', 'nitrogen_dioxide', 'ozone', 
                'temperature_2m', 'wind_speed_10m', 'relative_humidity_2m']
corr = processed_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Feature Engineering Results

In [ ]:
# Load featured data
featured_df = pd.read_csv(f'{DATA_DIR}/featured_aqi_data.csv', index_col=0, parse_dates=True)
print(f"Featured data shape: {featured_df.shape}")
print(f"\nFeature columns: {len(featured_df.columns)}")
print(featured_df.columns.tolist()[:20], "...")

## 4. Model Training

In [ ]:
# Prepare data for modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define features and target
target_col = 'aqi_target_24h'
target_cols = [col for col in featured_df.columns if 'target' in col]
feature_cols = [col for col in featured_df.columns 
                if col not in target_cols and col not in ['aqi', 'aqi_category']]

# Split data
train_mask = featured_df.index.year == 2024
val_mask = (featured_df.index.year == 2025) & (featured_df.index.month <= 3)
test_mask = (featured_df.index.year == 2025) & (featured_df.index.month > 3)

X_train = featured_df.loc[train_mask, feature_cols]
X_test = featured_df.loc[test_mask, feature_cols]
y_train = featured_df.loc[train_mask, target_col]
y_test = featured_df.loc[test_mask, target_col]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Train Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(scaler.transform(X_test))

print("Linear Regression Results:")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_lr):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.2f}")
print(f"  R2: {r2_score(y_test, y_pred_lr):.4f}")

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results:")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_rf):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f}")
print(f"  R2: {r2_score(y_test, y_pred_rf):.4f}")

## 5. Model Comparison

In [ ]:
# Model comparison
results = {
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_rf)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf))
    ],
    'R2': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_rf)
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

models = results_df['Model'].values
x = np.arange(len(models))

axes[0].bar(x, results_df['MAE'], color='steelblue')
axes[0].set_title('MAE Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=45)

axes[1].bar(x, results_df['RMSE'], color='coral')
axes[1].set_title('RMSE Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=45)

axes[2].bar(x, results_df['R2'], color='seagreen')
axes[2].set_title('R² Comparison')
axes[2].set_xticks(x)
axes[2].set_xticklabels(models, rotation=45)

plt.tight_layout()
plt.show()

## 6. Feature Importance (Random Forest)

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance.head(15)['feature'], importance.head(15)['importance'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Conclusion

### Key Findings:
1. **Data Characteristics**: PM2.5 shows clear daily and seasonal patterns
2. **Best Model**: Random Forest outperforms Linear Regression with lower MAE and RMSE
3. **Key Features**: Lag features and rolling statistics are the most important predictors
4. **Prediction Accuracy**: The model can predict AQI within acceptable error margins

### Recommendations:
- Use Random Forest for production forecasting
- Retrain models regularly with fresh data
- Consider ensemble methods for improved accuracy